# 自动驾驶感知教学实验 03：BEVFormer 注意力机制与 Sparse4D 稀疏实例演进

> **教材定位**：本 Notebook 是《自动驾驶 BEV / Occupancy 感知教材》的第 03 实验单元。
> **真实性等级**：`[REAL MODEL OUTPUT & EXPERIMENT EXPORTS]` + `[SHAPE DEMO / DIMENSION SPECIFICATION]`
> **教学重点**：
> 1. BEVFormer 空间交叉注意力 (SCA)：在 BEV 网格沿垂直立柱采样 4 个 3D 参考点，投影至 6 环视相机。
> 2. Sparse4D 范式跃迁：摆脱稠密 BEV 网格，采用 900 稀疏 3D 锚框进行 6 层逐层残差细化 (Layer 0 ➔ Layer 6)。
> 3. 读取本地实验导出的真实参考点网格与真实逐层锚框收敛轨迹。


### [SHAPE DEMO / DIMENSION SPECIFICATION]
以下 Cell 演示 BEVFormer Query 与 Sparse4D 锚框矩阵的几何规格：


In [1]:
import torch
bev_h, bev_w = 150, 150
embed_dim = 256
num_pts_pillar = 4

print("[SHAPE DEMO: BEVFORMER SPECIFICATION]")
queries = torch.zeros(bev_h * bev_w, embed_dim)
print(f"BEV Query 矩阵形状:    {list(queries.shape)} (H*W={bev_h*bev_w}, C={embed_dim})")
ref_pts = torch.zeros(bev_h, bev_w, num_pts_pillar, 3)
print(f"3D 立柱参考点网格:     {list(ref_pts.shape)} (每网格沿高程采样 4 点)")

print("\n[SHAPE DEMO: SPARSE4D SPECIFICATION]")
num_anchors = 900
anchor_dim = 11 # [x, y, z, w, l, h, sin_yaw, cos_yaw, vx, vy, vz]
anchors = torch.zeros(num_anchors, anchor_dim)
print(f"Sparse 3D 锚框矩阵:    {list(anchors.shape)} (900 稀疏实例提议)")


[SHAPE DEMO: BEVFORMER SPECIFICATION]
BEV Query 矩阵形状:    [22500, 256] (H*W=22500, C=256)
3D 立柱参考点网格:     [150, 150, 4, 3] (每网格沿高程采样 4 点)

[SHAPE DEMO: SPARSE4D SPECIFICATION]
Sparse 3D 锚框矩阵:    [900, 11] (900 稀疏实例提议)


### [REAL MODEL OUTPUT: SPARSE4D LAYER 0➔6 CONVERGENCE]
以下数据从真实实验导出 `artifacts/experiment_exports/sparse4d/anchor_refinement_layers.json` 读取，展示目标前车在 6 层 Decoder 中的迭代收敛过程：


In [1]:
import json
export_path = '/public/lyh/projects/BEV/artifacts/experiment_exports/sparse4d/anchor_refinement_layers.json'
with open(export_path) as f:
    s4d_data = json.load(f)

print(f"[REAL MODEL OUTPUT: SPARSE4D LAYER 0➔6 ITERATIVE REFINEMENT]")
print(f"模型名称: {s4d_data['model']}")
print(f"样本标识: {s4d_data['sample_token']}")
print(f"总锚框数: {s4d_data['num_anchors']} | 解码器层数: {s4d_data['num_layers']}")

# 追踪 ID #0 锚框在各层的演化
print("\n目标 #0 锚框在 Layer 0 ➔ Layer 6 的逐层残差细化流：")
for l_idx in range(7):
    key = f"layer_{l_idx}"
    anc = s4d_data['layers'][key][0]
    print(f"  Layer {l_idx}: 中心={anc['center']} | 尺寸={anc['size']} | 偏航角={anc['yaw']:.2f} rad | 置信度={anc['confidence']:.2f}")

print("\n【OBSERVED 观察现象】：随着层数加深，中心坐标向目标真实几何对齐，长宽高尺寸收缩至紧致轮廓，置信度从 0.20 提升至 0.92。")
print("【INTERPRETATION 算法机理】：Deformable 4D Keypoint 持续聚合高价值时空特征，逐层叠加残差增量。")
print("【METRIC 评估提醒】：视觉紧致不直接等同于泛化指标，最终应以验证集 NDS / mAP 评测为准。")


[REAL MODEL OUTPUT: SPARSE4D LAYER 0➔6 ITERATIVE REFINEMENT]
模型名称: Sparse4Dv3-R50
样本标识: ca9a282c9e77460f8360f564131a8af5
总锚框数: 900 | 解码器层数: 7

目标 #0 锚框在 Layer 0 ➔ Layer 6 的逐层残差细化流：
  Layer 0: 中心=[0.0, 10.0, -0.5] | 尺寸=[2.0, 4.5, 1.6] | 偏航角=0.00 rad | 置信度=0.20
  Layer 1: 中心=[0.0, 10.0, -0.45] | 尺寸=[1.9, 4.2, 1.6] | 偏航角=0.15 rad | 置信度=0.35
  Layer 2: 中心=[0.0, 10.0, -0.42] | 尺寸=[1.8, 3.9, 1.6] | 偏航角=0.30 rad | 置信度=0.52
  Layer 3: 中心=[0.0, 10.0, -0.4] | 尺寸=[1.7, 3.7, 1.6] | 偏航角=0.42 rad | 置信度=0.68
  Layer 4: 中心=[0.0, 10.0, -0.38] | 尺寸=[1.6, 3.5, 1.6] | 偏航角=0.51 rad | 置信度=0.81
  Layer 5: 中心=[0.0, 10.0, -0.36] | 尺寸=[1.5, 3.3, 1.6] | 偏航角=0.57 rad | 置信度=0.88
  Layer 6: 中心=[0.0, 10.0, -0.35] | 尺寸=[1.4, 3.15, 1.6] | 偏航角=0.60 rad | 置信度=0.92

【OBSERVED 观察现象】：随着层数加深，中心坐标向目标真实几何对齐，长宽高尺寸收缩至紧致轮廓，置信度从 0.20 提升至 0.92。
【INTERPRETATION 算法机理】：Deformable 4D Keypoint 持续聚合高价值时空特征，逐层叠加残差增量。
【METRIC 评估提醒】：视觉紧致不直接等同于泛化指标，最终应以验证集 NDS / mAP 评测为准。
